In [1]:
# ============================================================
# 1. IMPORT LIBRARY
# ============================================================
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from google.colab import drive

# mount drive
drive.mount('/content/drive')

# load data
makro_2010_2024 = pd.read_excel("/content/drive/My Drive/Data Inflasi, BI Rate, Kurs.xlsx")
birate = makro_2010_2024[['Tanggal', 'Data BI Rate']]
saham_groups = pd.read_excel("/content/drive/My Drive/Hasil Re-Cluster.xlsx")
return_aktual = pd.read_excel("/content/drive/My Drive/Data Return 2010-2025.xlsx")

# preview
birate.head()
saham_groups.head()
return_aktual

Mounted at /content/drive


,Tanggal,AALI,ACES,ADHI,ADRO,AKRA,AMRT,ANTM,ASII,ASRI,...,SMRA,SSIA,TINS,TKIM,TLKM,TPIA,TRAM,UNSP,UNTR,UNVR
0,2010-02-01,0.014568,-0.064079,-0.037741,-0.032261,-0.145712,0.162518,-0.023811,0.008310,0.221307,...,0.028171,0.000000,-0.034289,-0.021391,-0.119121,0.204794,-0.058269,-0.128617,0.017647,0.017544
1,2010-03-01,0.016394,0.187212,0.120628,0.068629,-0.092373,-0.116534,0.145507,0.144846,0.187212,...,0.142175,0.219860,0.099529,0.026668,-0.030583,-0.076961,-0.010050,-0.029853,0.067632,0.054982
2,2010-04-01,-0.102654,0.041797,0.293348,0.115512,0.102129,0.211844,0.020620,0.118048,0.270772,...,0.166127,0.075986,0.128255,0.051293,-0.025159,-0.040822,-0.030772,0.010050,0.060785,0.130956
3,2010-05-01,-0.111888,0.016960,-0.185717,-0.095310,-0.032768,-0.095310,0.259219,-0.088651,-0.234840,...,-0.230438,-0.050010,-0.193495,-0.133531,-0.012821,-0.100644,-0.010471,-0.261365,-0.060785,0.118986
4,2010-06-01,-0.003120,0.039665,0.235314,0.003835,0.088728,0.061731,0.008027,0.130779,0.073688,...,0.098846,0.000000,-0.034289,0.133531,-0.006473,-0.023811,-0.043017,-0.039740,0.039273,0.102646
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
186,2025-08-01,0.070952,-0.030240,0.109484,-0.052717,-0.075759,-0.053110,0.064539,0.075508,0.091808,...,0.083770,-0.105361,0.009852,0.157629,0.083243,-0.117109,0.000000,0.287682,0.010299,-0.005865
187,2025-09-01,0.113206,-0.058708,0.036368,-0.037740,0.012474,-0.130937,0.038715,0.048790,0.045722,...,-0.050354,-0.259590,0.453321,-0.031749,-0.022618,-0.065751,0.000000,0.024098,0.092886,0.045985
188,2025-10-01,-0.031364,0.032039,-0.066445,0.109199,0.008230,0.040615,-0.019170,0.079955,-0.099789,...,-0.083178,-0.145875,0.508955,0.000000,0.047856,-0.105720,0.000000,0.011834,0.027055,0.371176
189,2025-11-01,-0.029270,-0.060343,-0.046884,-0.040601,0.016261,-0.110348,-0.063249,0.063013,0.012270,...,-0.015424,0.183923,0.199649,0.082521,0.089345,0.063286,0.000000,0.192078,0.040078,0.007722


In [2]:
# ============================================================
# 2. MENYIAPKAN DATA RETURN HASIL PREDIKSI
# ============================================================
from functools import reduce

# mount drive
drive.mount('/content/drive')

# load data
hargasaham = pd.read_excel("/content/drive/My Drive/Data Harga 2010-2025.xlsx")
loop1 = pd.read_excel("/content/drive/My Drive/hasil_loop1/Excel/hasil_loop1.xlsx", sheet_name = "forecast_2025_lolos")
loop2 = pd.read_excel("/content/drive/My Drive/hasil_loop2/Excel/hasil_loop2.xlsx", sheet_name = "forecast_2025_lolos")
loop3 = pd.read_excel("/content/drive/My Drive/hasil_loop3/Excel/hasil_loop3.xlsx", sheet_name = "forecast_2025_lolos")
loop5 = pd.read_excel("/content/drive/My Drive/hasil_loop5/Excel/hasil_loop5.xlsx", sheet_name = "forecast_2025_lolos")
loop8 = pd.read_excel("/content/drive/My Drive/hasil_loop8/Excel/hasil_loop8.xlsx", sheet_name = "forecast_2025_lolos")

#ambil data hsitoris 2010-2024
harga_2010_2024 = hargasaham[
    (hargasaham["Tanggal"].dt.year >= 2010) &
    (hargasaham["Tanggal"].dt.year <= 2024)
].copy()

# simpan semua dataframe loop dalam list
list_loop = [loop1, loop2, loop3, loop5, loop8]

# gabungkan berdasarkan kolom Tanggal
harga_pred_2025 = reduce(
    lambda left, right: pd.merge(left, right, on="Tanggal", how="outer"),
    list_loop
)

# urutkan berdasarkan tanggal
harga_pred_2025 = harga_pred_2025.sort_values("Tanggal").reset_index(drop=True)

# ambil nama saham yang ada di harga_pred_2025
saham = [col for col in harga_pred_2025.columns if col != "Tanggal"]

# ambil hanya kolom Tanggal dan saham yang sesuai
harga_2010_2024 = harga_2010_2024[["Tanggal"] + saham].copy()

# urutkan berdasarkan tanggal
harga_2010_2024 = harga_2010_2024.sort_values("Tanggal").reset_index(drop=True)

# gabungkan data historis 2010-2024 dan prediksi 2025
harga_2010_2025 = pd.concat(
    [harga_2010_2024, harga_pred_2025],
    axis=0,
    ignore_index=True
)

# urutkan berdasarkan tanggal
harga_2010_2025 = harga_2010_2025.sort_values("Tanggal").reset_index(drop=True)

# siapkan data cluster
groups_df = saham_groups[["Saham", "Cluster"]].copy()

# daftar cluster
cluster_ids = sorted(groups_df["Cluster"].unique())

harga_2010_2025.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Tanggal,AALI,ACES,ADRO,AKRA,AMRT,ANTM,ASII,BBCA,BBNI,...,TRAM,UNTR,UNVR,KLBF,MNCN,SMGR,MAIN,TINS,SCMA,ADHI
0,2010-01-01,13786.080078,101.454010,379.139526,99.739410,72.379448,1296.536743,1840.732666,728.069580,530.491943,...,504.372253,6678.516602,1412.030273,232.721313,187.449341,5284.552734,86.714592,1010.241882,14.509254,244.556000
1,2010-02-01,13988.386719,95.156868,367.103424,86.215431,85.152252,1266.030029,1856.093506,709.867737,524.994507,...,475.822876,6797.420898,1437.021851,235.743652,203.749268,5020.324707,85.772049,976.188782,17.010847,235.498322
2,2010-03-01,14219.601562,114.747986,393.181824,78.608177,75.785507,1464.323730,2145.387695,800.876648,625.320740,...,471.064667,7273.042969,1518.244751,282.590149,264.874054,4822.153809,108.393242,1078.347656,18.011488,265.690399
3,2010-04-01,12832.326172,119.645737,441.326355,87.060661,93.667496,1494.830933,2414.201172,793.595703,714.652161,...,456.789978,7728.846680,1730.673706,313.569366,342.298798,5416.666016,91.427345,1225.911255,23.014679,356.266724
4,2010-05-01,11473.950195,121.692284,401.205872,84.254051,85.152252,1937.178345,2209.391113,808.157227,687.165588,...,452.031769,7273.042969,1949.351562,284.101349,317.848877,5581.807617,91.427345,1010.241882,22.514360,295.882477


In [3]:
# ============================================================
# 3. HITUNG LOG RETURN DARI harga_2010_2025
# ============================================================

# copy data agar aman
harga_df = harga_2010_2025.copy()

# pastikan Tanggal datetime
harga_df["Tanggal"] = pd.to_datetime(harga_df["Tanggal"])

# urutkan berdasarkan tanggal
harga_df = harga_df.sort_values("Tanggal").reset_index(drop=True)

# ambil kolom saham
stock_cols = [col for col in harga_df.columns if col != "Tanggal"]

# pastikan data harga saham numerik
for col in stock_cols:
    harga_df[col] = pd.to_numeric(harga_df[col], errors="coerce")

# hitung log return
return_pred = harga_df.copy()

return_pred[stock_cols] = np.log(
    harga_df[stock_cols] / harga_df[stock_cols].shift(1)
)

# hapus hanya baris pertama karena tidak punya return sebelumnya
# jangan pakai dropna(), karena bisa menghapus bulan lain jika ada NaN pada salah satu saham
return_pred = return_pred.iloc[1:].reset_index(drop=True)

# cek hasil
print("Jumlah baris return_pred:", len(return_pred))
print("Tanggal awal:", return_pred["Tanggal"].min())
print("Tanggal akhir:", return_pred["Tanggal"].max())

print("\nJumlah data tahun 2025:")
print(len(return_pred[return_pred["Tanggal"].dt.year == 2025]))

return_pred

Jumlah baris return_pred: 191
Tanggal awal: 2010-02-01 00:00:00
Tanggal akhir: 2025-12-01 00:00:00

Jumlah data tahun 2025:
12


/usr/local/lib/python3.12/dist-packages/pandas/core/internals/blocks.py:393: RuntimeWarning: invalid value encountered in log
  result = func(self.values, **kwargs)


,Tanggal,AALI,ACES,ADRO,AKRA,AMRT,ANTM,ASII,BBCA,BBNI,...,TRAM,UNTR,UNVR,KLBF,MNCN,SMGR,MAIN,TINS,SCMA,ADHI
0,2010-02-01,0.014568,-0.064079,-0.032261,-0.145712,0.162518,-0.023811,0.008310,-0.025318,-0.010417,...,-0.058269,0.017647,0.017544,0.012903,0.083382,-0.051293,-0.010929,-0.034289,0.159065,-0.037741
1,2010-03-01,0.016394,0.187212,0.068629,-0.092373,-0.116534,0.145507,0.144846,0.120628,0.174877,...,-0.010050,0.067632,0.054982,0.181253,0.262364,-0.040274,0.234073,0.099529,0.057159,0.120628
2,2010-04-01,-0.102654,0.041797,0.115512,0.102129,0.211844,0.020620,0.118048,-0.009133,0.133531,...,-0.030772,0.060785,0.130956,0.104023,0.256430,0.116260,-0.170221,0.128255,0.245122,0.293348
3,2010-05-01,-0.111888,0.016960,-0.095310,-0.032768,-0.095310,0.259219,-0.088651,0.018182,-0.039221,...,-0.010471,-0.060785,0.118986,-0.098690,-0.074108,0.030032,0.000000,-0.193495,-0.021979,-0.185717
4,2010-06-01,-0.003120,0.039665,0.003835,0.088728,0.061731,0.008027,0.130779,0.082286,-0.041894,...,-0.043017,0.039273,0.102646,0.110665,-0.080043,0.034888,0.000000,-0.034289,0.131974,0.235314
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
186,2025-08-01,0.019954,-0.030441,0.009631,0.010052,0.008271,0.011873,0.001474,0.007455,-0.003759,...,0.101680,0.006551,-0.020959,0.022807,0.552502,0.062734,-0.054200,0.016162,-0.023942,-0.043956
187,2025-09-01,0.008203,-0.031288,0.003223,0.005687,0.002481,0.012303,0.000654,0.000764,-0.010711,...,0.087274,0.002664,-0.007892,0.008659,0.394819,0.033254,-0.040510,0.012836,-0.013772,-0.188704
188,2025-10-01,0.004306,-0.030507,0.001926,0.003621,0.001281,0.006990,-0.000344,-0.000180,-0.009605,...,0.074050,0.002252,-0.001098,-0.002258,0.317435,-0.000982,-0.030516,0.009264,-0.024560,-0.396790
189,2025-11-01,-0.001038,-0.031082,0.000755,0.001813,0.000063,0.005856,-0.000972,-0.001183,-0.004763,...,0.063181,0.001464,0.004649,-0.011205,0.263876,-0.029352,-0.021124,0.006210,-0.028775,-0.831110


In [4]:
# ============================================================
# 4. HITUNG RISK-FREE BULANAN (SATU ANGKA)
# ============================================================
bi_df = birate.copy()
bi_rate_annual = bi_df["Data BI Rate"].dropna().values

# ubah BI rate tahunan menjadi ekuivalen bulanan
rf_monthly_series = (1 + bi_rate_annual) ** (1/12) - 1

# jumlah observasi rf bulanan
T_rf = len(rf_monthly_series)

# rata-rata geometrik bulanan
rf = np.prod(1 + rf_monthly_series) ** (1 / T_rf) - 1

print("Risk-free bulanan (rf) =", rf)

Risk-free bulanan (rf) = 0.004617798240782234


In [5]:
# ============================================================
# 5. SIAPKAN DATA RETURN DAN DATA CLUSTER
# ============================================================
# return_pred   = data return skenario 1, yaitu return hasil prediksi
# return_aktual = data return skenario 2, yaitu return aktual
# Kedua data memiliki kolom saham yang sama, tetapi berbeda pada tahun 2025.

# Load data cluster
saham_groups = pd.read_excel("/content/drive/My Drive/Hasil Re-Cluster.xlsx")

# Pastikan kolom Tanggal berbentuk datetime
return_pred["Tanggal"] = pd.to_datetime(return_pred["Tanggal"])
return_aktual["Tanggal"] = pd.to_datetime(return_aktual["Tanggal"])

# Ambil kolom saham yang sama pada return_pred dan return_aktual
stock_cols = [
    col for col in return_pred.columns
    if col != "Tanggal" and col in return_aktual.columns
]

# Samakan struktur kolom kedua skenario
return_pred = return_pred[["Tanggal"] + stock_cols].copy()
return_aktual = return_aktual[["Tanggal"] + stock_cols].copy()

# Ambil kolom Saham dan Cluster dari data cluster
groups_df = saham_groups[["Saham", "Cluster"]].copy()

# Gunakan hanya saham yang ada dalam data return
groups_df = groups_df[groups_df["Saham"].isin(stock_cols)].copy()

# Urutan saham mengikuti data cluster
stock_cols = groups_df["Saham"].tolist()

# Samakan lagi struktur return berdasarkan saham yang memiliki cluster
return_pred = return_pred[["Tanggal"] + stock_cols].copy()
return_aktual = return_aktual[["Tanggal"] + stock_cols].copy()

# Daftar cluster
cluster_ids = sorted(groups_df["Cluster"].unique())

print("Jumlah saham yang digunakan:", len(stock_cols))
print("Daftar cluster:", cluster_ids)

Jumlah saham yang digunakan: 63
Daftar cluster: [np.int64(1), np.int64(2)]


In [6]:
# ============================================================
# 6. FUNGSI OBJECTIVE OPTIMASI
# ============================================================
# Tahap 1:
#   Minimisasi VaR95 loss
#
# Tahap 2:
#   Maksimisasi Reward-to-VaR Ratio
#   dengan constraint:
#   VaR95 loss tahap 2 <= VaR95 loss minimum + toleransi
#
# Dengan demikian, bobot final tetap mempertahankan tingkat risiko
# minimum atau sangat dekat dengan risiko minimum, tetapi dipilih
# berdasarkan Reward-to-VaR Ratio terbaik.

def calc_var95_loss(weights, returns_matrix):
    # hitung return portofolio
    port_ret = np.dot(returns_matrix, weights)

    # quantile 5% return
    q05 = np.quantile(port_ret, 0.05)

    # VaR95 loss
    # 1e-12 hanya digunakan untuk stabilitas numerik dalam proses optimasi
    var95_loss = max(-q05, 1e-12)

    return var95_loss


def calc_portfolio_metrics(weights, returns_matrix, rf):
    # hitung return portofolio
    port_ret = np.dot(returns_matrix, weights)

    # mean return
    mean_return = np.mean(port_ret)

    # quantile 5% return
    q05 = np.quantile(port_ret, 0.05)

    # VaR95 loss
    var95_loss = max(-q05, 1e-12)

    # excess return
    excess_return = mean_return - rf

    # Reward-to-VaR Ratio
    reward_to_var = excess_return / var95_loss

    return mean_return, q05, var95_loss, excess_return, reward_to_var


def objective_min_var95(weights, returns_matrix):
    # objective tahap 1:
    # minimalkan VaR95 loss
    return calc_var95_loss(weights, returns_matrix)


def objective_neg_reward_to_var(weights, returns_matrix, rf):
    # hitung metrik portofolio
    mean_return, q05, var95_loss, excess_return, reward_to_var = calc_portfolio_metrics(
        weights,
        returns_matrix,
        rf
    )

    # penalti jika ratio tidak valid
    if not np.isfinite(reward_to_var):
        return 1e10

    # negatif karena scipy.optimize.minimize melakukan minimisasi
    # sehingga minimize(-reward_to_var) = maximize(reward_to_var)
    return -reward_to_var


def constraint_sum_weights(weights):
    # total bobot harus sama dengan 1
    return np.sum(weights) - 1


def optimize_min_var95_then_max_ratio_with_constraint(
    returns_matrix,
    rf,
    var_tolerance=1e-6
):
    # jumlah aset
    n_assets = returns_matrix.shape[1]

    # bobot awal sama rata
    w0 = np.repeat(1 / n_assets, n_assets)

    # batas bobot long-only
    bounds = tuple([(0, 1)] * n_assets)

    # constraint total bobot = 1
    constraints_stage1 = ({
        "type": "eq",
        "fun": constraint_sum_weights
    })

    # ========================================================
    # TAHAP 1: MINIMALKAN VaR95 LOSS
    # ========================================================

    result_stage1 = minimize(
        fun=objective_min_var95,
        x0=w0,
        args=(returns_matrix,),
        method="SLSQP",
        bounds=bounds,
        constraints=constraints_stage1
    )

    if not result_stage1.success:
        print("Warning tahap 1 minimisasi VaR95:", result_stage1.message)

    # bobot hasil minimisasi VaR95 loss
    weights_min_var95 = result_stage1.x

    # nilai VaR95 minimum
    var95_minimum = calc_var95_loss(
        weights_min_var95,
        returns_matrix
    )

    # ========================================================
    # TAHAP 2: MAKSIMALKAN REWARD-TO-VaR RATIO
    # DENGAN CONSTRAINT VaR95 LOSS TIDAK MELEBIHI VaR MINIMUM
    # ========================================================

    def constraint_var95_not_exceed_min(weights):
        current_var95 = calc_var95_loss(
            weights,
            returns_matrix
        )

        # scipy constraint inequality harus bernilai >= 0
        # artinya:
        # var95_minimum + var_tolerance - current_var95 >= 0
        return (var95_minimum + var_tolerance) - current_var95

    constraints_stage2 = (
        {
            "type": "eq",
            "fun": constraint_sum_weights
        },
        {
            "type": "ineq",
            "fun": constraint_var95_not_exceed_min
        }
    )

    result_stage2 = minimize(
        fun=objective_neg_reward_to_var,
        x0=weights_min_var95,
        args=(returns_matrix, rf),
        method="SLSQP",
        bounds=bounds,
        constraints=constraints_stage2
    )

    if not result_stage2.success:
        print("Warning tahap 2 maksimisasi Reward-to-VaR Ratio:", result_stage2.message)

    # bobot hasil maksimisasi Reward-to-VaR Ratio
    # dengan constraint VaR95 minimum
    weights_max_ratio = result_stage2.x

    # metrik tahap 1
    mean_1, q05_1, var95_1, excess_1, ratio_1 = calc_portfolio_metrics(
        weights_min_var95,
        returns_matrix,
        rf
    )

    # metrik tahap 2
    mean_2, q05_2, var95_2, excess_2, ratio_2 = calc_portfolio_metrics(
        weights_max_ratio,
        returns_matrix,
        rf
    )

    # ringkasan hasil optimasi
    optim_summary = {
        "Mean_Return_Tahap1_Min_VaR95": mean_1,
        "Quantile_5_Tahap1_Min_VaR95": q05_1,
        "VaR95_Loss_Tahap1_Minimum": var95_1,
        "Excess_Return_Tahap1": excess_1,
        "Reward_to_VaR_Tahap1": ratio_1,

        "Mean_Return_Tahap2_Max_Ratio": mean_2,
        "Quantile_5_Tahap2_Max_Ratio": q05_2,
        "VaR95_Loss_Tahap2_Max_Ratio": var95_2,
        "Excess_Return_Tahap2": excess_2,
        "Reward_to_VaR_Tahap2_Max_Ratio": ratio_2,

        "VaR95_Minimum_Batas": var95_minimum,
        "Var_Tolerance": var_tolerance
    }

    return weights_min_var95, weights_max_ratio, optim_summary

In [7]:
# ============================================================
# 7. SKENARIO 1 - OPTIMASI INTRA-KLASTER
# ============================================================
# Input return yang digunakan adalah return_pred.
#
# Tahap 1:
#   Minimisasi VaR95 loss
#
# Tahap 2:
#   Maksimisasi Reward-to-VaR Ratio
#   dengan constraint VaR95 loss tidak boleh melebihi VaR95 minimum.

returns_df_s1 = return_pred.copy()
R_s1 = returns_df_s1[stock_cols].copy()

intra_weights_minvar_dict_s1 = {}
intra_weights_ratio_dict_s1 = {}

cluster_return_ratio_dict_s1 = {}

for cl in cluster_ids:

    # ambil saham dalam cluster ke-cl
    stocks_in_cluster = groups_df.loc[
        groups_df["Cluster"] == cl,
        "Saham"
    ].tolist()

    # ambil data return saham dalam cluster
    cluster_returns_df = R_s1[stocks_in_cluster].copy()
    cluster_returns_df = cluster_returns_df.fillna(0)

    # matriks return cluster
    returns_matrix = cluster_returns_df.values

    # optimasi bertahap dengan constraint VaR minimum
    w_min_var95, w_max_ratio, optim_summary = optimize_min_var95_then_max_ratio_with_constraint(
        returns_matrix=returns_matrix,
        rf=rf,
        var_tolerance=1e-6
    )

    # return portofolio cluster hasil tahap akhir
    # digunakan sebagai input optimasi inter-klaster
    port_ret_ratio = np.dot(returns_matrix, w_max_ratio)

    # simpan bobot hasil minimisasi VaR95 loss
    intra_weights_minvar = pd.DataFrame({
        "Skenario": "Skenario1_Prediksi",
        "Tahap_Optimasi": "Minimisasi_VaR95_Loss",
        "Cluster": cl,
        "Saham": stocks_in_cluster,
        "Bobot_Intra_Min_VaR95": w_min_var95
    })

    # simpan bobot hasil maksimisasi Reward-to-VaR Ratio
    # dengan constraint VaR minimum
    intra_weights_ratio = pd.DataFrame({
        "Skenario": "Skenario1_Prediksi",
        "Tahap_Optimasi": "Maksimisasi_Reward_to_VaR_dengan_Batas_VaR_Minimum",
        "Cluster": cl,
        "Saham": stocks_in_cluster,
        "Bobot_Intra_Max_Ratio": w_max_ratio
    })

    intra_weights_minvar_dict_s1[cl] = intra_weights_minvar
    intra_weights_ratio_dict_s1[cl] = intra_weights_ratio
    cluster_return_ratio_dict_s1[cl] = port_ret_ratio

# gabungkan bobot intra-klaster
intra_weights_minvar_skenario1 = pd.concat(
    intra_weights_minvar_dict_s1.values(),
    ignore_index=True
)

intra_weights_ratio_skenario1 = pd.concat(
    intra_weights_ratio_dict_s1.values(),
    ignore_index=True
)

# urutkan bobot intra-klaster
intra_weights_minvar_skenario1 = intra_weights_minvar_skenario1.sort_values(
    ["Cluster", "Bobot_Intra_Min_VaR95"],
    ascending=[True, False]
).reset_index(drop=True)

intra_weights_ratio_skenario1 = intra_weights_ratio_skenario1.sort_values(
    ["Cluster", "Bobot_Intra_Max_Ratio"],
    ascending=[True, False]
).reset_index(drop=True)

print("================ BOBOT INTRA-KLASTER MINIMUM VaR95 SKENARIO 1 ================")
print(intra_weights_minvar_skenario1)

print("\n================ BOBOT INTRA-KLASTER MAXIMUM RATIO DENGAN BATAS VaR MINIMUM SKENARIO 1 ================")
print(intra_weights_ratio_skenario1)

Warning tahap 2 maksimisasi Reward-to-VaR Ratio: Iteration limit reached
Warning tahap 2 maksimisasi Reward-to-VaR Ratio: Iteration limit reached
================ BOBOT INTRA-KLASTER MINIMUM VaR95 SKENARIO 1 ================
              Skenario         Tahap_Optimasi  Cluster Saham  \
0   Skenario1_Prediksi  Minimisasi_VaR95_Loss        1  SGRO   
1   Skenario1_Prediksi  Minimisasi_VaR95_Loss        1  EMTK   
2   Skenario1_Prediksi  Minimisasi_VaR95_Loss        1  AMRT   
3   Skenario1_Prediksi  Minimisasi_VaR95_Loss        1  TRAM   
4   Skenario1_Prediksi  Minimisasi_VaR95_Loss        1  ANTM   
..                 ...                    ...      ...   ...   
58  Skenario1_Prediksi  Minimisasi_VaR95_Loss        2  BBTN   
59  Skenario1_Prediksi  Minimisasi_VaR95_Loss        2  BMTR   
60  Skenario1_Prediksi  Minimisasi_VaR95_Loss        2  SMGR   
61  Skenario1_Prediksi  Minimisasi_VaR95_Loss        2  BSDE   
62  Skenario1_Prediksi  Minimisasi_VaR95_Loss        2  BDMN   

    Bo

In [8]:
# ============================================================
# 8. SKENARIO 1 - BENTUK RETURN PORTOFOLIO PER CLUSTER
# ============================================================
# Return portofolio cluster ini hanya digunakan sebagai input
# untuk optimasi inter-klaster.
# Tidak perlu disimpan ke Excel jika hanya ingin menyimpan bobot.

cluster_returns_ratio_skenario1 = pd.DataFrame({
    "Tanggal": returns_df_s1["Tanggal"]
})

for cl in cluster_ids:
    cluster_returns_ratio_skenario1[f"Cluster_{cl}"] = cluster_return_ratio_dict_s1[cl]

print("================ RETURN CLUSTER MAXIMUM RATIO SKENARIO 1 ================")
print(cluster_returns_ratio_skenario1.head())

================ RETURN CLUSTER MAXIMUM RATIO SKENARIO 1 ================
     Tanggal  Cluster_1  Cluster_2
0 2010-02-01   0.000951  -0.008257
1 2010-03-01   0.014388   0.063164
2 2010-04-01   0.079482   0.091656
3 2010-05-01  -0.066781  -0.048590
4 2010-06-01   0.018035   0.089455


In [9]:
# ============================================================
# 9. SKENARIO 1 - OPTIMASI INTER-KLASTER
# ============================================================
# Input inter-klaster menggunakan return cluster hasil tahap akhir intra-klaster.
#
# Tahap 1:
#   Minimisasi VaR95 loss antar-klaster
#
# Tahap 2:
#   Maksimisasi Reward-to-VaR Ratio antar-klaster
#   dengan constraint VaR95 loss tidak boleh melebihi VaR95 minimum.

cluster_ret_cols_s1 = [
    col for col in cluster_returns_ratio_skenario1.columns
    if col != "Tanggal"
]

cluster_ret_matrix_s1 = cluster_returns_ratio_skenario1[
    cluster_ret_cols_s1
].fillna(0).values

# optimasi antar-cluster bertahap dengan constraint VaR minimum
w_cluster_minvar_s1, w_cluster_ratio_s1, inter_optim_summary_s1 = optimize_min_var95_then_max_ratio_with_constraint(
    returns_matrix=cluster_ret_matrix_s1,
    rf=rf,
    var_tolerance=1e-6
)

# tabel bobot antar-cluster hasil minimisasi VaR95
cluster_weights_minvar_skenario1 = pd.DataFrame({
    "Skenario": "Skenario1_Prediksi",
    "Tahap_Optimasi": "Minimisasi_VaR95_Loss",
    "Cluster": cluster_ids,
    "Bobot_Antar_Cluster_Min_VaR95": w_cluster_minvar_s1
})

# tabel bobot antar-cluster hasil maksimisasi ratio
# dengan constraint VaR minimum
cluster_weights_ratio_skenario1 = pd.DataFrame({
    "Skenario": "Skenario1_Prediksi",
    "Tahap_Optimasi": "Maksimisasi_Reward_to_VaR_dengan_Batas_VaR_Minimum",
    "Cluster": cluster_ids,
    "Bobot_Antar_Cluster_Max_Ratio": w_cluster_ratio_s1
})

# urutkan bobot antar-cluster
cluster_weights_minvar_skenario1 = cluster_weights_minvar_skenario1.sort_values(
    "Bobot_Antar_Cluster_Min_VaR95",
    ascending=False
).reset_index(drop=True)

cluster_weights_ratio_skenario1 = cluster_weights_ratio_skenario1.sort_values(
    "Bobot_Antar_Cluster_Max_Ratio",
    ascending=False
).reset_index(drop=True)

print("================ BOBOT INTER-KLASTER MINIMUM VaR95 SKENARIO 1 ================")
print(cluster_weights_minvar_skenario1)

print("\n================ BOBOT INTER-KLASTER MAXIMUM RATIO DENGAN BATAS VaR MINIMUM SKENARIO 1 ================")
print(cluster_weights_ratio_skenario1)

================ BOBOT INTER-KLASTER MINIMUM VaR95 SKENARIO 1 ================
             Skenario         Tahap_Optimasi  Cluster  \
0  Skenario1_Prediksi  Minimisasi_VaR95_Loss        2   
1  Skenario1_Prediksi  Minimisasi_VaR95_Loss        1   

   Bobot_Antar_Cluster_Min_VaR95  
0                            1.0  
1                            0.0  

================ BOBOT INTER-KLASTER MAXIMUM RATIO DENGAN BATAS VaR MINIMUM SKENARIO 1 ================
             Skenario                                     Tahap_Optimasi  \
0  Skenario1_Prediksi  Maksimisasi_Reward_to_VaR_dengan_Batas_VaR_Min...   
1  Skenario1_Prediksi  Maksimisasi_Reward_to_VaR_dengan_Batas_VaR_Min...   

   Cluster  Bobot_Antar_Cluster_Max_Ratio  
0        2                            1.0  
1        1                            0.0  


In [10]:
# ============================================================
# 10. SKENARIO 1 - BENTUK BOBOT FINAL SAHAM
# ============================================================
# Bobot final saham dihitung dengan:
#
# Bobot_Final = Bobot_Intra_Max_Ratio x Bobot_Antar_Cluster_Max_Ratio
#
# Catatan:
# Bobot Max Ratio di sini sudah dibatasi oleh constraint VaR minimum.

final_weights_list_s1 = []

for cl in cluster_ids:

    # ambil bobot intra-klaster hasil tahap akhir
    intra_df = intra_weights_ratio_dict_s1[cl].copy()

    # ambil bobot antar-cluster hasil tahap akhir
    w_cluster = cluster_weights_ratio_skenario1.loc[
        cluster_weights_ratio_skenario1["Cluster"] == cl,
        "Bobot_Antar_Cluster_Max_Ratio"
    ].values[0]

    # hitung bobot final saham
    intra_df["Bobot_Antar_Cluster"] = w_cluster
    intra_df["Bobot_Final"] = (
        intra_df["Bobot_Intra_Max_Ratio"] *
        intra_df["Bobot_Antar_Cluster"]
    )

    final_weights_list_s1.append(intra_df)

# gabungkan bobot final saham
final_weights_skenario1 = pd.concat(
    final_weights_list_s1,
    ignore_index=True
)

# urutkan dari bobot terbesar
final_weights_skenario1 = final_weights_skenario1.sort_values(
    "Bobot_Final",
    ascending=False
).reset_index(drop=True)

# normalisasi agar total bobot final = 1
final_weights_skenario1["Bobot_Final"] = (
    final_weights_skenario1["Bobot_Final"] /
    final_weights_skenario1["Bobot_Final"].sum()
)

# pembulatan tampilan
final_weights_skenario1["Bobot_Intra_Max_Ratio"] = final_weights_skenario1["Bobot_Intra_Max_Ratio"].round(10)
final_weights_skenario1["Bobot_Antar_Cluster"] = final_weights_skenario1["Bobot_Antar_Cluster"].round(10)
final_weights_skenario1["Bobot_Final"] = final_weights_skenario1["Bobot_Final"].round(10)

# koreksi selisih pembulatan agar total tetap 1
selisih_s1 = 1 - final_weights_skenario1["Bobot_Final"].sum()
idx_max_s1 = final_weights_skenario1["Bobot_Final"].idxmax()
final_weights_skenario1.loc[idx_max_s1, "Bobot_Final"] += selisih_s1

# pembulatan akhir
final_weights_skenario1["Bobot_Final"] = final_weights_skenario1["Bobot_Final"].round(10)

print("================ BOBOT FINAL SAHAM SKENARIO 1 ================")
print(final_weights_skenario1)

print(
    "\nJumlah total bobot final skenario 1 =",
    f"{final_weights_skenario1['Bobot_Final'].sum():.10f}"
)

================ BOBOT FINAL SAHAM SKENARIO 1 ================
              Skenario                                     Tahap_Optimasi  \
0   Skenario1_Prediksi  Maksimisasi_Reward_to_VaR_dengan_Batas_VaR_Min...   
1   Skenario1_Prediksi  Maksimisasi_Reward_to_VaR_dengan_Batas_VaR_Min...   
2   Skenario1_Prediksi  Maksimisasi_Reward_to_VaR_dengan_Batas_VaR_Min...   
3   Skenario1_Prediksi  Maksimisasi_Reward_to_VaR_dengan_Batas_VaR_Min...   
4   Skenario1_Prediksi  Maksimisasi_Reward_to_VaR_dengan_Batas_VaR_Min...   
..                 ...                                                ...   
58  Skenario1_Prediksi  Maksimisasi_Reward_to_VaR_dengan_Batas_VaR_Min...   
59  Skenario1_Prediksi  Maksimisasi_Reward_to_VaR_dengan_Batas_VaR_Min...   
60  Skenario1_Prediksi  Maksimisasi_Reward_to_VaR_dengan_Batas_VaR_Min...   
61  Skenario1_Prediksi  Maksimisasi_Reward_to_VaR_dengan_Batas_VaR_Min...   
62  Skenario1_Prediksi  Maksimisasi_Reward_to_VaR_dengan_Batas_VaR_Min...   

    Cluster 

In [11]:
# ============================================================
# 11. SKENARIO 2 - OPTIMASI INTRA-KLASTER
# ============================================================
# Input return yang digunakan adalah return_aktual.
#
# Tahap 1:
#   Minimisasi VaR95 loss
#
# Tahap 2:
#   Maksimisasi Reward-to-VaR Ratio
#   dengan constraint VaR95 loss tidak boleh melebihi VaR95 minimum.

returns_df_s2 = return_aktual.copy()
R_s2 = returns_df_s2[stock_cols].copy()

intra_weights_minvar_dict_s2 = {}
intra_weights_ratio_dict_s2 = {}

cluster_return_ratio_dict_s2 = {}

for cl in cluster_ids:

    # ambil saham dalam cluster ke-cl
    stocks_in_cluster = groups_df.loc[
        groups_df["Cluster"] == cl,
        "Saham"
    ].tolist()

    # ambil data return saham dalam cluster
    cluster_returns_df = R_s2[stocks_in_cluster].copy()
    cluster_returns_df = cluster_returns_df.fillna(0)

    # matriks return cluster
    returns_matrix = cluster_returns_df.values

    # optimasi bertahap dengan constraint VaR minimum
    w_min_var95, w_max_ratio, optim_summary = optimize_min_var95_then_max_ratio_with_constraint(
        returns_matrix=returns_matrix,
        rf=rf,
        var_tolerance=1e-6
    )

    # return portofolio cluster hasil tahap akhir
    # digunakan sebagai input optimasi inter-klaster
    port_ret_ratio = np.dot(returns_matrix, w_max_ratio)

    # simpan bobot hasil minimisasi VaR95 loss
    intra_weights_minvar = pd.DataFrame({
        "Skenario": "Skenario2_Aktual",
        "Tahap_Optimasi": "Minimisasi_VaR95_Loss",
        "Cluster": cl,
        "Saham": stocks_in_cluster,
        "Bobot_Intra_Min_VaR95": w_min_var95
    })

    # simpan bobot hasil maksimisasi Reward-to-VaR Ratio
    # dengan constraint VaR minimum
    intra_weights_ratio = pd.DataFrame({
        "Skenario": "Skenario2_Aktual",
        "Tahap_Optimasi": "Maksimisasi_Reward_to_VaR_dengan_Batas_VaR_Minimum",
        "Cluster": cl,
        "Saham": stocks_in_cluster,
        "Bobot_Intra_Max_Ratio": w_max_ratio
    })

    intra_weights_minvar_dict_s2[cl] = intra_weights_minvar
    intra_weights_ratio_dict_s2[cl] = intra_weights_ratio
    cluster_return_ratio_dict_s2[cl] = port_ret_ratio

# gabungkan bobot intra-klaster
intra_weights_minvar_skenario2 = pd.concat(
    intra_weights_minvar_dict_s2.values(),
    ignore_index=True
)

intra_weights_ratio_skenario2 = pd.concat(
    intra_weights_ratio_dict_s2.values(),
    ignore_index=True
)

# urutkan bobot intra-klaster
intra_weights_minvar_skenario2 = intra_weights_minvar_skenario2.sort_values(
    ["Cluster", "Bobot_Intra_Min_VaR95"],
    ascending=[True, False]
).reset_index(drop=True)

intra_weights_ratio_skenario2 = intra_weights_ratio_skenario2.sort_values(
    ["Cluster", "Bobot_Intra_Max_Ratio"],
    ascending=[True, False]
).reset_index(drop=True)

print("================ BOBOT INTRA-KLASTER MINIMUM VaR95 SKENARIO 2 ================")
print(intra_weights_minvar_skenario2)

print("\n================ BOBOT INTRA-KLASTER MAXIMUM RATIO DENGAN BATAS VaR MINIMUM SKENARIO 2 ================")
print(intra_weights_ratio_skenario2)

Warning tahap 2 maksimisasi Reward-to-VaR Ratio: Iteration limit reached
================ BOBOT INTRA-KLASTER MINIMUM VaR95 SKENARIO 2 ================
            Skenario         Tahap_Optimasi  Cluster Saham  \
0   Skenario2_Aktual  Minimisasi_VaR95_Loss        1  ANTM   
1   Skenario2_Aktual  Minimisasi_VaR95_Loss        1  ISAT   
2   Skenario2_Aktual  Minimisasi_VaR95_Loss        1  EMTK   
3   Skenario2_Aktual  Minimisasi_VaR95_Loss        1  TRAM   
4   Skenario2_Aktual  Minimisasi_VaR95_Loss        1  TKIM   
..               ...                    ...      ...   ...   
58  Skenario2_Aktual  Minimisasi_VaR95_Loss        2  BMRI   
59  Skenario2_Aktual  Minimisasi_VaR95_Loss        2  BBTN   
60  Skenario2_Aktual  Minimisasi_VaR95_Loss        2  MNCN   
61  Skenario2_Aktual  Minimisasi_VaR95_Loss        2  BMTR   
62  Skenario2_Aktual  Minimisasi_VaR95_Loss        2  CTRA   

    Bobot_Intra_Min_VaR95  
0            1.504903e-01  
1            8.666169e-02  
2            6.9609

In [12]:
# ============================================================
# 12. SKENARIO 2 - BENTUK RETURN PORTOFOLIO PER CLUSTER
# ============================================================
# Return portofolio cluster ini hanya digunakan sebagai input
# untuk optimasi inter-klaster.
# Tidak perlu disimpan ke Excel jika hanya ingin menyimpan bobot.

cluster_returns_ratio_skenario2 = pd.DataFrame({
    "Tanggal": returns_df_s2["Tanggal"]
})

for cl in cluster_ids:
    cluster_returns_ratio_skenario2[f"Cluster_{cl}"] = cluster_return_ratio_dict_s2[cl]

print("================ RETURN CLUSTER MAXIMUM RATIO SKENARIO 2 ================")
print(cluster_returns_ratio_skenario2.head())

================ RETURN CLUSTER MAXIMUM RATIO SKENARIO 2 ================
     Tanggal  Cluster_1  Cluster_2
0 2010-02-01   0.034383   0.006921
1 2010-03-01  -0.001657   0.056935
2 2010-04-01   0.176209   0.089334
3 2010-05-01  -0.081337  -0.047835
4 2010-06-01   0.047978   0.086465


In [13]:
# ============================================================
# 13. SKENARIO 2 - OPTIMASI INTER-KLASTER
# ============================================================
# Input inter-klaster menggunakan return cluster hasil tahap akhir intra-klaster.
#
# Tahap 1:
#   Minimisasi VaR95 loss antar-klaster
#
# Tahap 2:
#   Maksimisasi Reward-to-VaR Ratio antar-klaster
#   dengan constraint VaR95 loss tidak boleh melebihi VaR95 minimum.

cluster_ret_cols_s2 = [
    col for col in cluster_returns_ratio_skenario2.columns
    if col != "Tanggal"
]

cluster_ret_matrix_s2 = cluster_returns_ratio_skenario2[
    cluster_ret_cols_s2
].fillna(0).values

# optimasi antar-cluster bertahap dengan constraint VaR minimum
w_cluster_minvar_s2, w_cluster_ratio_s2, inter_optim_summary_s2 = optimize_min_var95_then_max_ratio_with_constraint(
    returns_matrix=cluster_ret_matrix_s2,
    rf=rf,
    var_tolerance=1e-6
)

# tabel bobot antar-cluster hasil minimisasi VaR95
cluster_weights_minvar_skenario2 = pd.DataFrame({
    "Skenario": "Skenario2_Aktual",
    "Tahap_Optimasi": "Minimisasi_VaR95_Loss",
    "Cluster": cluster_ids,
    "Bobot_Antar_Cluster_Min_VaR95": w_cluster_minvar_s2
})

# tabel bobot antar-cluster hasil maksimisasi ratio
# dengan constraint VaR minimum
cluster_weights_ratio_skenario2 = pd.DataFrame({
    "Skenario": "Skenario2_Aktual",
    "Tahap_Optimasi": "Maksimisasi_Reward_to_VaR_dengan_Batas_VaR_Minimum",
    "Cluster": cluster_ids,
    "Bobot_Antar_Cluster_Max_Ratio": w_cluster_ratio_s2
})

# urutkan bobot antar-cluster
cluster_weights_minvar_skenario2 = cluster_weights_minvar_skenario2.sort_values(
    "Bobot_Antar_Cluster_Min_VaR95",
    ascending=False
).reset_index(drop=True)

cluster_weights_ratio_skenario2 = cluster_weights_ratio_skenario2.sort_values(
    "Bobot_Antar_Cluster_Max_Ratio",
    ascending=False
).reset_index(drop=True)

print("================ BOBOT INTER-KLASTER MINIMUM VaR95 SKENARIO 2 ================")
print(cluster_weights_minvar_skenario2)

print("\n================ BOBOT INTER-KLASTER MAXIMUM RATIO DENGAN BATAS VaR MINIMUM SKENARIO 2 ================")
print(cluster_weights_ratio_skenario2)

================ BOBOT INTER-KLASTER MINIMUM VaR95 SKENARIO 2 ================
           Skenario         Tahap_Optimasi  Cluster  \
0  Skenario2_Aktual  Minimisasi_VaR95_Loss        2   
1  Skenario2_Aktual  Minimisasi_VaR95_Loss        1   

   Bobot_Antar_Cluster_Min_VaR95  
0                   1.000000e+00  
1                   6.661338e-16  

================ BOBOT INTER-KLASTER MAXIMUM RATIO DENGAN BATAS VaR MINIMUM SKENARIO 2 ================
           Skenario                                     Tahap_Optimasi  \
0  Skenario2_Aktual  Maksimisasi_Reward_to_VaR_dengan_Batas_VaR_Min...   
1  Skenario2_Aktual  Maksimisasi_Reward_to_VaR_dengan_Batas_VaR_Min...   

   Cluster  Bobot_Antar_Cluster_Max_Ratio  
0        2                       0.999964  
1        1                       0.000036  


In [14]:
# ============================================================
# 14. SKENARIO 2 - BENTUK BOBOT FINAL SAHAM
# ============================================================
# Bobot final saham dihitung dengan:
#
# Bobot_Final = Bobot_Intra_Max_Ratio x Bobot_Antar_Cluster_Max_Ratio
#
# Catatan:
# Bobot Max Ratio di sini sudah dibatasi oleh constraint VaR minimum.

final_weights_list_s2 = []

for cl in cluster_ids:

    # ambil bobot intra-klaster hasil tahap akhir
    intra_df = intra_weights_ratio_dict_s2[cl].copy()

    # ambil bobot antar-cluster hasil tahap akhir
    w_cluster = cluster_weights_ratio_skenario2.loc[
        cluster_weights_ratio_skenario2["Cluster"] == cl,
        "Bobot_Antar_Cluster_Max_Ratio"
    ].values[0]

    # hitung bobot final saham
    intra_df["Bobot_Antar_Cluster"] = w_cluster
    intra_df["Bobot_Final"] = (
        intra_df["Bobot_Intra_Max_Ratio"] *
        intra_df["Bobot_Antar_Cluster"]
    )

    final_weights_list_s2.append(intra_df)

# gabungkan bobot final saham
final_weights_skenario2 = pd.concat(
    final_weights_list_s2,
    ignore_index=True
)

# urutkan dari bobot terbesar
final_weights_skenario2 = final_weights_skenario2.sort_values(
    "Bobot_Final",
    ascending=False
).reset_index(drop=True)

# normalisasi agar total bobot final = 1
final_weights_skenario2["Bobot_Final"] = (
    final_weights_skenario2["Bobot_Final"] /
    final_weights_skenario2["Bobot_Final"].sum()
)

# pembulatan tampilan
final_weights_skenario2["Bobot_Intra_Max_Ratio"] = final_weights_skenario2["Bobot_Intra_Max_Ratio"].round(10)
final_weights_skenario2["Bobot_Antar_Cluster"] = final_weights_skenario2["Bobot_Antar_Cluster"].round(10)
final_weights_skenario2["Bobot_Final"] = final_weights_skenario2["Bobot_Final"].round(10)

# koreksi selisih pembulatan agar total tetap 1
selisih_s2 = 1 - final_weights_skenario2["Bobot_Final"].sum()
idx_max_s2 = final_weights_skenario2["Bobot_Final"].idxmax()
final_weights_skenario2.loc[idx_max_s2, "Bobot_Final"] += selisih_s2

# pembulatan akhir
final_weights_skenario2["Bobot_Final"] = final_weights_skenario2["Bobot_Final"].round(10)

print("================ BOBOT FINAL SAHAM SKENARIO 2 ================")
print(final_weights_skenario2)

print(
    "\nJumlah total bobot final skenario 2 =",
    f"{final_weights_skenario2['Bobot_Final'].sum():.10f}"
)

================ BOBOT FINAL SAHAM SKENARIO 2 ================
            Skenario                                     Tahap_Optimasi  \
0   Skenario2_Aktual  Maksimisasi_Reward_to_VaR_dengan_Batas_VaR_Min...   
1   Skenario2_Aktual  Maksimisasi_Reward_to_VaR_dengan_Batas_VaR_Min...   
2   Skenario2_Aktual  Maksimisasi_Reward_to_VaR_dengan_Batas_VaR_Min...   
3   Skenario2_Aktual  Maksimisasi_Reward_to_VaR_dengan_Batas_VaR_Min...   
4   Skenario2_Aktual  Maksimisasi_Reward_to_VaR_dengan_Batas_VaR_Min...   
..               ...                                                ...   
58  Skenario2_Aktual  Maksimisasi_Reward_to_VaR_dengan_Batas_VaR_Min...   
59  Skenario2_Aktual  Maksimisasi_Reward_to_VaR_dengan_Batas_VaR_Min...   
60  Skenario2_Aktual  Maksimisasi_Reward_to_VaR_dengan_Batas_VaR_Min...   
61  Skenario2_Aktual  Maksimisasi_Reward_to_VaR_dengan_Batas_VaR_Min...   
62  Skenario2_Aktual  Maksimisasi_Reward_to_VaR_dengan_Batas_VaR_Min...   

    Cluster Saham  Bobot_Intra_Max_R

In [15]:
# ============================================================
# 15. SAVE BOBOT SAJA - SKENARIO 1
# ============================================================

output_path_s1 = "/content/drive/My Drive/Bobot Portofolio Skenario 1.xlsx"

with pd.ExcelWriter(output_path_s1, engine="openpyxl") as writer:

    # bobot intra-klaster hasil minimisasi VaR95 loss
    intra_weights_minvar_skenario1.to_excel(
        writer,
        sheet_name="Intra_Min_VaR95",
        index=False
    )

    # bobot intra-klaster hasil maksimisasi Reward-to-VaR Ratio
    # dengan constraint VaR minimum
    intra_weights_ratio_skenario1.to_excel(
        writer,
        sheet_name="Intra_Max_Ratio",
        index=False
    )

    # bobot inter-klaster hasil minimisasi VaR95 loss
    cluster_weights_minvar_skenario1.to_excel(
        writer,
        sheet_name="Inter_Min_VaR95",
        index=False
    )

    # bobot inter-klaster hasil maksimisasi Reward-to-VaR Ratio
    # dengan constraint VaR minimum
    cluster_weights_ratio_skenario1.to_excel(
        writer,
        sheet_name="Inter_Max_Ratio",
        index=False
    )

    # bobot final saham
    final_weights_skenario1.to_excel(
        writer,
        sheet_name="Bobot_Final",
        index=False
    )

print("File bobot Skenario 1 berhasil disimpan ke:")
print(output_path_s1)

File bobot Skenario 1 berhasil disimpan ke:
/content/drive/My Drive/Bobot Portofolio Skenario 1.xlsx


In [16]:
# ============================================================
# 16. SAVE BOBOT SAJA - SKENARIO 2
# ============================================================

output_path_s2 = "/content/drive/My Drive/Bobot Portofolio Skenario 2.xlsx"

with pd.ExcelWriter(output_path_s2, engine="openpyxl") as writer:

    # bobot intra-klaster hasil minimisasi VaR95 loss
    intra_weights_minvar_skenario2.to_excel(
        writer,
        sheet_name="Intra_Min_VaR95",
        index=False
    )

    # bobot intra-klaster hasil maksimisasi Reward-to-VaR Ratio
    # dengan constraint VaR minimum
    intra_weights_ratio_skenario2.to_excel(
        writer,
        sheet_name="Intra_Max_Ratio",
        index=False
    )

    # bobot inter-klaster hasil minimisasi VaR95 loss
    cluster_weights_minvar_skenario2.to_excel(
        writer,
        sheet_name="Inter_Min_VaR95",
        index=False
    )

    # bobot inter-klaster hasil maksimisasi Reward-to-VaR Ratio
    # dengan constraint VaR minimum
    cluster_weights_ratio_skenario2.to_excel(
        writer,
        sheet_name="Inter_Max_Ratio",
        index=False
    )

    # bobot final saham
    final_weights_skenario2.to_excel(
        writer,
        sheet_name="Bobot_Final",
        index=False
    )

print("File bobot Skenario 2 berhasil disimpan ke:")
print(output_path_s2)

File bobot Skenario 2 berhasil disimpan ke:
/content/drive/My Drive/Bobot Portofolio Skenario 2.xlsx


In [17]:
# ============================================================
# HITUNG RETURN PORTOFOLIO 2025 UNTUK KOMBINASI BOBOT DAN RETURN
# ============================================================

# pastikan tanggal datetime
return_pred["Tanggal"] = pd.to_datetime(return_pred["Tanggal"])
return_aktual["Tanggal"] = pd.to_datetime(return_aktual["Tanggal"])

# ambil data tahun 2025
return_pred_2025 = return_pred[
    return_pred["Tanggal"].dt.year == 2025
].copy().reset_index(drop=True)

return_aktual_2025 = return_aktual[
    return_aktual["Tanggal"].dt.year == 2025
].copy().reset_index(drop=True)

# ============================================================
# 1. SUSUN VEKTOR BOBOT SKENARIO 1 DAN SKENARIO 2
# ============================================================

# Bobot Skenario 1 = bobot berbasis return prediksi
weight_map_s1 = dict(zip(
    final_weights_skenario1["Saham"],
    final_weights_skenario1["Bobot_Final"]
))

final_weight_vector_s1 = np.array([
    weight_map_s1.get(stock, 0) for stock in stock_cols
])

# Bobot Skenario 2 = bobot berbasis return aktual
weight_map_s2 = dict(zip(
    final_weights_skenario2["Saham"],
    final_weights_skenario2["Bobot_Final"]
))

final_weight_vector_s2 = np.array([
    weight_map_s2.get(stock, 0) for stock in stock_cols
])

# ============================================================
# 2. AMBIL MATRIKS RETURN 2025
# ============================================================

R_pred_2025 = return_pred_2025[stock_cols].fillna(0).values
R_aktual_2025 = return_aktual_2025[stock_cols].fillna(0).values

# ============================================================
# 3. HITUNG RETURN PORTOFOLIO BULANAN 2025
# ============================================================
# Karena return saham yang digunakan adalah log return,
# maka hasil return portofolio bulanan di bawah ini diperlakukan
# sebagai log return portofolio bulanan.

# 1. Bobot Skenario 1 x return prediksi
ret_bobot_s1_return_pred = np.dot(
    R_pred_2025,
    final_weight_vector_s1
)

# 2. Bobot Skenario 1 x return aktual
ret_bobot_s1_return_aktual = np.dot(
    R_aktual_2025,
    final_weight_vector_s1
)

# 3. Bobot Skenario 2 x return aktual
ret_bobot_s2_return_aktual = np.dot(
    R_aktual_2025,
    final_weight_vector_s2
)

# 4. Bobot Skenario 2 x return prediksi
# Bagian ini opsional, hanya jika ingin perbandingan silang tambahan
ret_bobot_s2_return_pred = np.dot(
    R_pred_2025,
    final_weight_vector_s2
)

# ============================================================
# 4. GABUNGKAN HASIL RETURN PORTOFOLIO BULANAN 2025
# ============================================================

portfolio_comparison_2025 = pd.DataFrame({
    "Tanggal": return_pred_2025["Tanggal"].values,

    "Bobot_Skenario1_x_Return_Prediksi": ret_bobot_s1_return_pred,
    "Bobot_Skenario1_x_Return_Aktual": ret_bobot_s1_return_aktual,
    "Bobot_Skenario2_x_Return_Aktual": ret_bobot_s2_return_aktual,
    "Bobot_Skenario2_x_Return_Prediksi": ret_bobot_s2_return_pred
})

print("================ RETURN PORTOFOLIO BULANAN 2025 ================")
print(portfolio_comparison_2025)

# ============================================================
# 5. HITUNG CUMULATIVE LOG RETURN 2025
# ============================================================
# Cumulative Log Return 2025:
# jumlah seluruh log return bulanan selama tahun 2025.

summary_return_2025 = pd.DataFrame({
    "Portofolio": [
        "Bobot Skenario 1 x Return Prediksi",
        "Bobot Skenario 1 x Return Aktual",
        "Bobot Skenario 2 x Return Aktual",
        "Bobot Skenario 2 x Return Prediksi"
    ],

    "Cumulative_Log_Return_2025": [
        np.sum(ret_bobot_s1_return_pred),
        np.sum(ret_bobot_s1_return_aktual),
        np.sum(ret_bobot_s2_return_aktual),
        np.sum(ret_bobot_s2_return_pred)
    ],

    "Mean_Log_Return_Bulanan_2025": [
        np.mean(ret_bobot_s1_return_pred),
        np.mean(ret_bobot_s1_return_aktual),
        np.mean(ret_bobot_s2_return_aktual),
        np.mean(ret_bobot_s2_return_pred)
    ]
})

print("\n================ RINGKASAN LOG RETURN PORTOFOLIO 2025 ================")
print(summary_return_2025)

================ RETURN PORTOFOLIO BULANAN 2025 ================
      Tanggal  Bobot_Skenario1_x_Return_Prediksi  \
0  2025-01-01                           0.032396   
1  2025-02-01                           0.027461   
2  2025-03-01                           0.020986   
3  2025-04-01                           0.012543   
4  2025-05-01                           0.008100   
5  2025-06-01                           0.005037   
6  2025-07-01                           0.001841   
7  2025-08-01                           0.002606   
8  2025-09-01                          -0.001033   
9  2025-10-01                          -0.002048   
10 2025-11-01                          -0.002827   
11 2025-12-01                          -0.002267   

    Bobot_Skenario1_x_Return_Aktual  Bobot_Skenario2_x_Return_Aktual  \
0                         -0.035888                        -0.019836   
1                         -0.077811                        -0.047839   
2                          0.006783       

In [18]:
# ============================================================
# HITUNG RETURN PORTOFOLIO 2025 UNTUK KOMBINASI BOBOT DAN RETURN
# ============================================================

# pastikan tanggal datetime
return_pred["Tanggal"] = pd.to_datetime(return_pred["Tanggal"])
return_aktual["Tanggal"] = pd.to_datetime(return_aktual["Tanggal"])

# ambil data tahun 2025
return_pred_2025 = return_pred[
    return_pred["Tanggal"].dt.year == 2025
].copy().reset_index(drop=True)

return_aktual_2025 = return_aktual[
    return_aktual["Tanggal"].dt.year == 2025
].copy().reset_index(drop=True)

# ============================================================
# 1. SUSUN VEKTOR BOBOT SKENARIO 1 DAN SKENARIO 2
# ============================================================

# Bobot Skenario 1 = bobot berbasis return prediksi
weight_map_s1 = dict(zip(
    final_weights_skenario1["Saham"],
    final_weights_skenario1["Bobot_Final"]
))

final_weight_vector_s1 = np.array([
    weight_map_s1.get(stock, 0) for stock in stock_cols
])

# Bobot Skenario 2 = bobot berbasis return aktual
weight_map_s2 = dict(zip(
    final_weights_skenario2["Saham"],
    final_weights_skenario2["Bobot_Final"]
))

final_weight_vector_s2 = np.array([
    weight_map_s2.get(stock, 0) for stock in stock_cols
])

# ============================================================
# 2. AMBIL MATRIKS RETURN 2025
# ============================================================

R_pred_2025 = return_pred_2025[stock_cols].fillna(0).values
R_aktual_2025 = return_aktual_2025[stock_cols].fillna(0).values

# ============================================================
# 3. HITUNG RETURN PORTOFOLIO BULANAN 2025
# ============================================================
# Karena return saham yang digunakan adalah log return,
# maka hasil return portofolio bulanan di bawah ini diperlakukan
# sebagai log return portofolio bulanan.

# 1. Bobot Skenario 1 x return prediksi
ret_bobot_s1_return_pred = np.dot(
    R_pred_2025,
    final_weight_vector_s1
)

# 2. Bobot Skenario 1 x return aktual
ret_bobot_s1_return_aktual = np.dot(
    R_aktual_2025,
    final_weight_vector_s1
)

# 3. Bobot Skenario 2 x return aktual
ret_bobot_s2_return_aktual = np.dot(
    R_aktual_2025,
    final_weight_vector_s2
)

# 4. Bobot Skenario 2 x return prediksi
# Bagian ini opsional, hanya jika ingin perbandingan silang tambahan
ret_bobot_s2_return_pred = np.dot(
    R_pred_2025,
    final_weight_vector_s2
)

# ============================================================
# 4. GABUNGKAN HASIL RETURN PORTOFOLIO BULANAN 2025
# ============================================================

portfolio_comparison_2025 = pd.DataFrame({
    "Tanggal": return_pred_2025["Tanggal"].values,

    "Bobot_Skenario1_x_Return_Prediksi": ret_bobot_s1_return_pred,
    "Bobot_Skenario1_x_Return_Aktual": ret_bobot_s1_return_aktual,
    "Bobot_Skenario2_x_Return_Aktual": ret_bobot_s2_return_aktual,
    "Bobot_Skenario2_x_Return_Prediksi": ret_bobot_s2_return_pred
})

print("================ RETURN PORTOFOLIO BULANAN 2025 ================")
print(portfolio_comparison_2025)

# ============================================================
# 5. FUNGSI HITUNG METRIK RETURN DAN RISIKO
# ============================================================
# Cumulative Log Return 2025:
# jumlah seluruh log return bulanan selama tahun 2025.
#
# VaR95 Loss:
# dihitung dari quantile 5% return bulanan tahun 2025.
#
# Reward-to-VaR Ratio:
# dihitung dari mean log return bulanan, rf bulanan, dan VaR95 loss.
#
# Catatan:
# Untuk evaluasi/laporan, VaR95 loss menggunakan max(-q05, 0).
# Jika VaR95 loss = 0, Reward-to-VaR Ratio dibuat NaN.

def hitung_metrik_portofolio_2025(nama_portofolio, return_series, rf):
    ret = np.array(return_series, dtype=float)

    cumulative_log_return = np.sum(ret)
    mean_log_return = np.mean(ret)

    q05 = np.quantile(ret, 0.05)
    var95_loss = max(-q05, 0)

    if var95_loss == 0:
        reward_to_var_ratio = np.nan
    else:
        reward_to_var_ratio = (mean_log_return - rf) / var95_loss

    return {
        "Portofolio": nama_portofolio,
        "Cumulative_Log_Return_2025": cumulative_log_return,
        "Mean_Log_Return_Bulanan_2025": mean_log_return,
        "Quantile_5_Return": q05,
        "VaR95_Loss": var95_loss,
        "Reward_to_VaR_Ratio": reward_to_var_ratio
    }

# ============================================================
# 6. HITUNG METRIK UNTUK KEEMPAT KOMBINASI PORTOFOLIO
# ============================================================

summary_return_risk_2025 = pd.DataFrame([
    hitung_metrik_portofolio_2025(
        "Prediksi-Prediksi",
        ret_bobot_s1_return_pred,
        rf
    ),
    hitung_metrik_portofolio_2025(
        "Prediksi-Aktual",
        ret_bobot_s1_return_aktual,
        rf
    ),
    hitung_metrik_portofolio_2025(
        "Aktual-Aktual",
        ret_bobot_s2_return_aktual,
        rf
    ),
    hitung_metrik_portofolio_2025(
        "Aktual-Prediksi",
        ret_bobot_s2_return_pred,
        rf
    )
])

print("\n================ RINGKASAN RETURN DAN RISIKO PORTOFOLIO 2025 ================")
print(summary_return_risk_2025)

================ RETURN PORTOFOLIO BULANAN 2025 ================
      Tanggal  Bobot_Skenario1_x_Return_Prediksi  \
0  2025-01-01                           0.032396   
1  2025-02-01                           0.027461   
2  2025-03-01                           0.020986   
3  2025-04-01                           0.012543   
4  2025-05-01                           0.008100   
5  2025-06-01                           0.005037   
6  2025-07-01                           0.001841   
7  2025-08-01                           0.002606   
8  2025-09-01                          -0.001033   
9  2025-10-01                          -0.002048   
10 2025-11-01                          -0.002827   
11 2025-12-01                          -0.002267   

    Bobot_Skenario1_x_Return_Aktual  Bobot_Skenario2_x_Return_Aktual  \
0                         -0.035888                        -0.019836   
1                         -0.077811                        -0.047839   
2                          0.006783       